In [1]:
from ngsolve import *
from netgen.geom2d import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np

# ------------------------------------------------------------
# 1. GEOMETRIE UND MESH
# ------------------------------------------------------------
H, L = 4.0, 28.0
cx, cy, R = 7.0, 0.0, 0.5
nu = 1e-3  # Viskosität

# Geometrie
rect = MoveTo(0, -H/2).Rectangle(L, H).Face()
rect.edges.Min(X).name = "inlet"
rect.edges.Max(X).name = "outlet"
rect.edges.Min(Y).name = "walls"
rect.edges.Max(Y).name = "walls"

cyl = Circle((cx, cy), R).Face()
cyl.edges.name = "obstacle"

shape = rect - cyl

# Mesh
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.4))
mesh.Refine()

# ------------------------------------------------------------
# 2. FE-RÄUME (Taylor-Hood)
# ------------------------------------------------------------
V = VectorH1(mesh, order=2, dirichlet="inlet|walls|obstacle")
Q = H1(mesh, order=1)
X = FESpace([V, Q], constraints=[])

(u, p) = X.TrialFunction()
(v, q) = X.TestFunction()

gfu = GridFunction(X)
gfu_u, gfu_p = gfu.components

# ------------------------------------------------------------
# 3. RANDBEDINGUNGEN (Inlet Profil)
# ------------------------------------------------------------
Umax = 0.1
uin_x = Umax * (1 - (2*y/H)**2)
uin = CoefficientFunction((uin_x, 0))
gfu_u.Set(uin, definedon=mesh.Boundaries("inlet"))

# ------------------------------------------------------------
# 4. STOKES LÖSEN (Anfangslösung)
# ------------------------------------------------------------
print("=== LÖSE STOKES (Anfangslösung) ===")

alpha = 1e-10  # Druckstabilisierung
gamma = 1e-2   # Grad-Div-Stabilisierung

a_stokes = BilinearForm(X)
a_stokes += 2*nu*InnerProduct(Sym(Grad(u)), Sym(Grad(v))) * dx
a_stokes += gamma * div(u) * div(v) * dx
a_stokes += -div(v) * p * dx
a_stokes += -div(u) * q * dx
a_stokes += alpha * p * q * dx

L_stokes = LinearForm(X)  # Keine Volumenkräfte

a_stokes.Assemble()
L_stokes.Assemble()

inv_stokes = a_stokes.mat.Inverse(X.FreeDofs())
res = L_stokes.vec - a_stokes.mat * gfu.vec
gfu.vec.data += inv_stokes * res

print(f"Stokes gelöst: ||u|| = {Norm(gfu_u.vec):.6f}")

Draw(gfu.components[0],mesh)
Draw(gfu.components[1],mesh)


=== LÖSE STOKES (Anfangslösung) ===
Stokes gelöst: ||u|| = 4.167074


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [2]:


# ------------------------------------------------------------
# 5. ZEITINTEGRATION FÜR NAVIER-STOKES VORBEREITEN
# ------------------------------------------------------------
dt = 0.05
t_end = 5.0
t = 0.0

# Massenmatrix
m = BilinearForm(X)
m += InnerProduct(u, v) * dx
m.Assemble()

# Implizite Matrix (Zeit + Stokes)
A_dt = BilinearForm(X)
A_dt += (1/dt) * InnerProduct(u, v) * dx
A_dt += 2*nu*InnerProduct(Sym(Grad(u)), Sym(Grad(v))) * dx  # Stokes-Anteil
A_dt += gamma * div(u) * div(v) * dx
A_dt += -div(v) * p * dx
A_dt += -div(u) * q * dx
A_dt += alpha * p * q * dx
A_dt.Assemble()

invA_dt = A_dt.mat.Inverse(X.FreeDofs())

# Konvektionsterm (nonassemble für Effizienz)
conv = BilinearForm(X, nonassemble=True)
conv += -InnerProduct(Grad(u) * u, v) * dx  # -(u·∇)u · v

# ------------------------------------------------------------
# 6. LIVE-VISUALISIERUNG VORBEREITEN
# ------------------------------------------------------------
print("\n=== STARTE NAVIER-STOKES SIMULATION ===")
print(f"Zeitschritt dt = {dt}, Endzeit t_end = {t_end}")

# Szenen für Live-Visualisierung erstellen
scene_u = Draw(gfu_u, mesh, "velocity", 
               min=0, max=0.15, autoscale=False,
               vectors={"grid_size": 30})

scene_p = Draw(gfu_p, mesh, "pressure",
               min=-0.02, max=0.02, autoscale=False)

# Für Animation speichern
gfut = GridFunction(V, multidim=0)  # Für Geschwindigkeitsanimation
vel_save_interval = 20  # Alle 20 Zeitschritte speichern

# ------------------------------------------------------------
# 7. ZEITINTEGRATION MIT LIVE-VISUALISIERUNG
# ------------------------------------------------------------
gfu_old = GridFunction(X)
gfu_old.vec.data = gfu.vec  # Stokes als Startwert

step = 0
save_counter = 0

while t < t_end:
    # --------------------------------------------------------
    # a) Rechte Seite berechnen
    # --------------------------------------------------------
    # Zeitanteil: M * u_old / dt
    rhs = (1/dt) * (m.mat * gfu_old.vec)
    
    # Konvektionsterm: -(u_old·∇)u_old
    conv_term = conv.Apply(gfu_old.vec)
    rhs += conv_term
    
    # --------------------------------------------------------
    # b) Lineares System lösen
    # --------------------------------------------------------
    gfu.vec.data = invA_dt * rhs
    
    # --------------------------------------------------------
    # c) Druck eindeutig machen
    # --------------------------------------------------------
    p_vec = gfu_p.vec.FV().NumPy()
    if len(p_vec) > 0:
        p_mean = np.mean(p_vec)
        p_vec[:] = p_vec[:] - p_mean
    
    # --------------------------------------------------------
    # d) Konvergenzüberwachung
    # --------------------------------------------------------
    div_u = div(gfu_u)
    div_norm = sqrt(Integrate(div_u**2, mesh))
    
    # Geschwindigkeitsänderung
    du_norm = Norm(gfu.vec - gfu_old.vec)
    
    # --------------------------------------------------------
    # e) VISUALISIERUNG AKTUALISIEREN (alle 5 Schritte)
    # --------------------------------------------------------
    if step % 5 == 0:
        scene_u.Redraw()
        scene_p.Redraw()
        print(f"t = {t:.3f}: ||div u|| = {div_norm:.2e}, Δu = {du_norm:.2e}")
    
    # --------------------------------------------------------
    # f) Für Animation speichern (alle 20 Schritte)
    # --------------------------------------------------------
    if step % vel_save_interval == 0:
        gfut.AddMultiDimComponent(gfu_u.vec)
        save_counter += 1
        print(f"  -> Gespeichert für Animation (Frame {save_counter})")
    
    # --------------------------------------------------------
    # g) Update für nächsten Zeitschritt
    # --------------------------------------------------------
    gfu_old.vec.data = gfu.vec
    t += dt
    step += 1

# ------------------------------------------------------------
# 8. FINALE AUSGABE
# ------------------------------------------------------------
print("\n=== SIMULATION ABGESCHLOSSEN ===")
print(f"Endzeit: t = {t:.3f}")
print(f"Anzahl Zeitschritte: {step}")
print(f"Animation: {save_counter} Frames gespeichert")

# Geschwindigkeitsmaximum
u_array = gfu_u.vec.FV().NumPy()
if len(u_array) > 0:
    u_max = np.max(np.sqrt(u_array[0::2]**2 + u_array[1::2]**2))
    print(f"Maximale Geschwindigkeit: {u_max:.4f}")

# ------------------------------------------------------------
# 9. ANIMATION ERSTELLEN UND ANZEIGEN
# ------------------------------------------------------------
print("\n=== ERSTELLE ANIMATION ===")

# Animation der gespeicherten Geschwindigkeitsfelder
if gfut.dim > 0:
    scene_anim = Draw(gfut, mesh, 
                      interpolate_multidim=True, 
                      animate=True,
                      min=0, max=0.15,
                      vectors={"grid_size": 30})
    print("Animation wird angezeigt...")
else:
    print("Keine Frames für Animation gespeichert!")

# ------------------------------------------------------------
# 10. FINALE VISUALISIERUNGEN
# ------------------------------------------------------------
print("\n=== FINALE VISUALISIERUNGEN ===")

# 1. Geschwindigkeitsbetrag
vel_mag = sqrt(gfu_u[0]**2 + gfu_u[1]**2)
Draw(vel_mag, mesh, "Velocity Magnitude", 
     min=0, max=0.15)

# 2. Vorticity (Wirbelstärke)
vorticity = grad(gfu_u[1])[0] - grad(gfu_u[0])[1]
Draw(vorticity, mesh, "Vorticity",
     min=-1, max=1)

# 3. Streamlines
Draw(gfu_u, mesh, "Streamlines",
     vectors={"grid_size": 40})

# 4. Druckverteilung (farbig)
Draw(gfu_p, mesh, "Pressure Distribution",
     deformation=gfu_u,  # Verformung für bessere Sichtbarkeit
     scale=10)  # Skalierung der Verformung

# ------------------------------------------------------------
# 11. MASSENERHALTUNG PRÜFEN
# ------------------------------------------------------------
print("\n=== MASSENERHALTUNG ===")
n = specialcf.normal(2)  # Normalenvektor

# Massenfluss an Einlass
inlet_flux = Integrate(gfu_u * n, mesh, 
                       definedon=mesh.Boundaries("inlet"))
# Massenfluss an Auslass
outlet_flux = Integrate(gfu_u * n, mesh,
                        definedon=mesh.Boundaries("outlet"))

print(f"Massenfluss Einlass:  {inlet_flux:.6f}")
print(f"Massenfluss Auslass:  {outlet_flux:.6f}")
print(f"Relativer Fehler:     {abs(inlet_flux - outlet_flux)/abs(inlet_flux)*100:.2f}%")

# ------------------------------------------------------------
# 12. ZUSÄTZLICHE ANALYSE: DRUCKDIFFERENZ VOR/NACH ZYLINDER
# ------------------------------------------------------------
print("\n=== DRUCKANALYSE ===")

# Druck am Einlass (Mittelwert)
p_inlet = Integrate(gfu_p, mesh, 
                    definedon=mesh.Boundaries("inlet")) / \
          Integrate(1, mesh, definedon=mesh.Boundaries("inlet"))

# Druck am Auslass (Mittelwert)
p_outlet = Integrate(gfu_p, mesh,
                     definedon=mesh.Boundaries("outlet")) / \
           Integrate(1, mesh, definedon=mesh.Boundaries("outlet"))

print(f"Druck Einlass (mittel):  {p_inlet:.6f}")
print(f"Druck Auslass (mittel):  {p_outlet:.6f}")
print(f"Druckdifferenz:          {p_inlet - p_outlet:.6f}")

print("\n=== ALLES FERTIG ===")


=== STARTE NAVIER-STOKES SIMULATION ===
Zeitschritt dt = 0.05, Endzeit t_end = 5.0


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

t = 0.000: ||div u|| = 5.37e-01, Δu = 6.46e+02
  -> Gespeichert für Animation (Frame 1)
t = 0.250: ||div u|| = 1.67e-01, Δu = 3.58e-01
t = 0.500: ||div u|| = 8.70e-02, Δu = 2.13e-01
t = 0.750: ||div u|| = 5.93e-02, Δu = 1.60e-01
t = 1.000: ||div u|| = 4.61e-02, Δu = 1.27e-01
  -> Gespeichert für Animation (Frame 2)
t = 1.250: ||div u|| = 3.82e-02, Δu = 1.05e-01
t = 1.500: ||div u|| = 3.28e-02, Δu = 8.85e-02
t = 1.750: ||div u|| = 2.89e-02, Δu = 7.59e-02
t = 2.000: ||div u|| = 2.60e-02, Δu = 6.61e-02
  -> Gespeichert für Animation (Frame 3)
t = 2.250: ||div u|| = 2.37e-02, Δu = 5.82e-02
t = 2.500: ||div u|| = 2.18e-02, Δu = 5.18e-02
t = 2.750: ||div u|| = 2.03e-02, Δu = 4.64e-02
t = 3.000: ||div u|| = 1.90e-02, Δu = 4.19e-02
  -> Gespeichert für Animation (Frame 4)
t = 3.250: ||div u|| = 1.79e-02, Δu = 3.80e-02
t = 3.500: ||div u|| = 1.69e-02, Δu = 3.47e-02
t = 3.750: ||div u|| = 1.61e-02, Δu = 3.17e-02
t = 4.000: ||div u|| = 1.53e-02, Δu = 2.91e-02
  -> Gespeichert für Animation (Frame

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Animation wird angezeigt...

=== FINALE VISUALISIERUNGEN ===


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

NgException: Operator grad not overloaded for CF ngfem::ComponentCoefficientFunction